# ECAPA scores for the Voice Banking report
Runs the SAME model and audio preprocessing as your app, here in Colab (internet + PyTorch included), and produces two small JSON files. You then build the report on your PC - no model needed there.

1. Runtime -> Change runtime type -> CPU is fine (GPU also works).
2. Run the cells from top to bottom. Cell 1 asks you to upload `Voice-Banking-Assistant-v17-READY.zip`.
3. The last cell downloads `scores_public.json` and `scores_own.json`.

In [ ]:
# 1) Upload the project zip
from google.colab import files
import zipfile
up = files.upload()
z = [n for n in up if n.lower().endswith('.zip')][0]
zipfile.ZipFile(z).extractall('/content')
%cd /content/voice-banking-assistant

In [ ]:
# 2) Libraries (PyTorch is already installed in Colab)
!pip -q install speechbrain soundfile scipy

In [ ]:
# 3) Download LibriSpeech dev-clean-2 (~126 MB, 40 speakers) and cut 4 s clips like the app records
!python ml/eval/prepare_public_dataset.py

**Optional - bigger, better test (LibriSpeech train-clean-100, 251 speakers, 6.3 GB):** replace cell 3's command with the line below. Colab downloads it in a few minutes and the script reads it as a stream (about 60 speakers are used), so the extra disk use is small. More speakers = a more trustworthy FAR.

`!python ml/eval/prepare_public_dataset.py --set train-clean-100 --speakers 60`

In [ ]:
# 4) Scores on the public dataset (a few minutes on CPU)
!python ml/eval/make_scores.py --dir data/public_samples --out reports/scores_public.json

In [ ]:
# 5) Scores on YOUR 18 recordings (6 people x 3)
!python ml/eval/make_scores.py --own samples/own_18 --out reports/scores_own.json

In [ ]:
# 6) Paper files (CSV, LaTeX tables, PDF/PNG graphs, confidence intervals, speaker-disjoint dev/test split)
!python ml/eval/paper_export.py --scores reports/scores_public.json --out paper/public --seed 1
!zip -qr paper_public.zip paper/public

# 7) Download everything
files.download('paper_public.zip')
files.download('reports/scores_public.json')
files.download('reports/scores_own.json')

Next, on your PC: drag each JSON onto `REPORT_FROM_SCORES.bat` (or run `node scripts\make_report.js --scores scores_public.json`).

**Kaggle instead of Colab:** New Notebook -> Settings -> Internet On -> upload the zip as a dataset, copy it to `/kaggle/working`, unzip, then run the same commands (replace `!pip` cell as is).